# Attestor 4.3 — Inference Benchmark on Kaggle
# Compare Qwythos-9B base vs Qwythos-9B + trained LoRA adapter
# on 5 security analysis tasks.
# **Runtime**: GPU T4 x2. Enable GPU in Settings > Accelerator.

In [ ]:
# Cell 1: Install dependencies (same as training notebook)
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
gpu_cap = result.stdout.strip().split('\n')[0]
print(f'GPU compute capability: {gpu_cap}')

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/transformers.git',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft', 'accelerate', 'bitsandbytes', 'sentencepiece', 'protobuf',
])

try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'flash-linear-attention', 'causal-conv1d',
    ])
    print('DeltaNet kernels installed.')
except subprocess.CalledProcessError:
    print('DeltaNet kernels not available — PyTorch fallback will be used.')

print('Dependencies installed.')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram = props.total_mem if hasattr(props, 'total_mem') else props.total_memory
    print(f'  GPU {i}: {props.name} — {vram / 1024**3:.1f} GiB')

In [ ]:
# Cell 3: Benchmark prompts — same 5 security tasks used for local Ollama baseline
import json, time

BENCHMARK_PROMPTS = [
    {
        'id': 'sqli_detect',
        'task': 'Identify the vulnerability in this code and give the CWE ID:\n\nquery = "SELECT * FROM users WHERE name = \'%s\'" % request.args.get(\'name\')\ncursor.execute(query)',
        'expected_keywords': ['SQL injection', 'CWE-89', 'parameterized', 'prepared statement'],
        'category': 'detection',
    },
    {
        'id': 'xss_exploit',
        'task': 'Write a proof-of-concept exploit for this reflected XSS:\n\n@app.route(\'/search\')\ndef search():\n    q = request.args.get(\'q\', \'\')\n    return f\'<h1>Results for {q}</h1>\'',
        'expected_keywords': ['<script>', 'alert', 'document.cookie', 'XSS', 'CWE-79'],
        'category': 'exploit',
    },
    {
        'id': 'path_traversal',
        'task': 'This endpoint has a vulnerability. What is it, what is the CWE, and how do you fix it?\n\n@app.route(\'/download\')\ndef download():\n    filename = request.args.get(\'file\')\n    return send_file(f\'/uploads/{filename}\')',
        'expected_keywords': ['path traversal', 'CWE-22', 'directory traversal', 'os.path.basename', 'sanitize'],
        'category': 'detection',
    },
    {
        'id': 'hardening',
        'task': 'Write a hardened version of this Flask login endpoint that prevents SQL injection:\n\ndef login():\n    user = request.form[\'username\']\n    pw = request.form[\'password\']\n    cur.execute("SELECT * FROM users WHERE user=\'%s\' AND pass=\'%s\'" % (user, pw))\n    return \'OK\' if cur.fetchone() else \'Fail\'',
        'expected_keywords': ['parameterized', '?', '%s', 'execute(', 'bcrypt', 'hash'],
        'category': 'hardening',
    },
    {
        'id': 'taint_analysis',
        'task': 'Trace the taint flow in this code. Where does user input reach a dangerous sink?\n\nname = request.cookies.get(\'username\')\ntemplate = \'<div class="greeting">Hello, \' + name + \'</div>\'\nreturn render_template_string(template)',
        'expected_keywords': ['SSTI', 'template injection', 'cookie', 'render_template_string', 'CWE-94', 'Jinja'],
        'category': 'analysis',
    },
]

def score_response(response, expected_keywords):
    rl = response.lower()
    hits = [kw for kw in expected_keywords if kw.lower() in rl]
    misses = [kw for kw in expected_keywords if kw.lower() not in rl]
    return len(hits) / len(expected_keywords), hits, misses

print(f'{len(BENCHMARK_PROMPTS)} benchmark prompts loaded.')

In [ ]:
# Cell 4: Load Qwythos-9B base model (4-bit quantized)
import os, torch, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
    from transformers import BloomPreTrainedModel
except Exception:
    import transformers
    transformers.BloomPreTrainedModel = type('BloomPreTrainedModel', (), {})

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'empero-ai/Qwythos-9B-Claude-Mythos-5-1M'

SYSTEM_PROMPT = (
    'You are Owen Coder 4.3, a Mythos-class security-focused code analysis model '
    'trained on the Attestor engine. You detect vulnerabilities, write exploit PoCs, '
    'perform taint analysis, and generate hardening recommendations. You reason step '
    'by step before answering. Be precise, technical, and exhaustive. Reference exact '
    'line numbers and CWE IDs. No refusals.'
)

print(f'Loading tokenizer from {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model (4-bit)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True)
except (ValueError, KeyError, TypeError):
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True)

model.eval()
print(f'Model loaded: {model.num_parameters():,} params')
print(f'Device map: {model.hf_device_map if hasattr(model, "hf_device_map") else "single GPU"}')

In [ ]:
# Cell 5: Inference helper
import torch

def generate_response(model, tokenizer, prompt, max_new_tokens=1024, temperature=0.1):
    formatted = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{prompt}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(formatted, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

# Quick smoke test
test = generate_response(model, tokenizer, 'What is CWE-79?', max_new_tokens=100)
print(f'Smoke test: {test[:200]}')

In [ ]:
# Cell 6: Run benchmark — Qwythos-9B BASE (no adapter)
import gc

print('=' * 60)
print('  BENCHMARK: Qwythos-9B BASE (no adapter)')
print('=' * 60)

base_results = []
for p in BENCHMARK_PROMPTS:
    print(f'\n--- [{p["id"]}] {p["category"]} ---')
    gc.collect()
    torch.cuda.empty_cache()
    
    t0 = time.time()
    try:
        resp = generate_response(model, tokenizer, p['task'])
    except Exception as e:
        resp = f'[ERROR: {e}]'
    elapsed = time.time() - t0
    
    sc, hits, misses = score_response(resp, p['expected_keywords'])
    base_results.append({
        'id': p['id'],
        'category': p['category'],
        'score': sc,
        'time': elapsed,
        'hits': hits,
        'misses': misses,
        'response': resp[:500],
        'resp_len': len(resp),
    })
    print(f'  Score: {sc:.0%} ({len(hits)}/{len(p["expected_keywords"])}) in {elapsed:.1f}s')
    print(f'  Hits: {hits}')
    print(f'  Misses: {misses}')
    print(f'  Response preview: {resp[:150]}...')

avg_base = sum(r['score'] for r in base_results) / len(base_results)
print(f'\n>>> BASE average: {avg_base:.0%}')

In [ ]:
# Cell 7: Load trained LoRA adapter
from peft import PeftModel
import os

# Check for adapter in Kaggle dataset or working directory
ADAPTER_PATHS = [
    '/kaggle/input/attestor-43-lora/adapter_config.json',
    '/kaggle/input/attestor-43-adapter/adapter_config.json',
    '/kaggle/working/attestor-43-lora-final/adapter_config.json',
]

adapter_dir = None
for p in ADAPTER_PATHS:
    if os.path.exists(p):
        adapter_dir = os.path.dirname(p)
        break

if adapter_dir is None:
    # Search /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files:
            adapter_dir = root
            break

if adapter_dir is None:
    print('ERROR: Adapter not found!')
    print('Upload the attestor-43-lora adapter as a Kaggle dataset.')
    print('Files needed: adapter_config.json, adapter_model.safetensors')
else:
    print(f'Found adapter at: {adapter_dir}')
    print('Files:', os.listdir(adapter_dir))
    
    # Load adapter on top of base model
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    print('LoRA adapter loaded successfully!')

In [ ]:
# Cell 8: Run benchmark — Qwythos-9B + LoRA ADAPTER
import gc

print('=' * 60)
print('  BENCHMARK: Qwythos-9B + TRAINED LoRA ADAPTER')
print('=' * 60)

adapter_results = []
for p in BENCHMARK_PROMPTS:
    print(f'\n--- [{p["id"]}] {p["category"]} ---')
    gc.collect()
    torch.cuda.empty_cache()
    
    t0 = time.time()
    try:
        resp = generate_response(model, tokenizer, p['task'])
    except Exception as e:
        resp = f'[ERROR: {e}]'
    elapsed = time.time() - t0
    
    sc, hits, misses = score_response(resp, p['expected_keywords'])
    adapter_results.append({
        'id': p['id'],
        'category': p['category'],
        'score': sc,
        'time': elapsed,
        'hits': hits,
        'misses': misses,
        'response': resp[:500],
        'resp_len': len(resp),
    })
    print(f'  Score: {sc:.0%} ({len(hits)}/{len(p["expected_keywords"])}) in {elapsed:.1f}s')
    print(f'  Hits: {hits}')
    print(f'  Misses: {misses}')
    print(f'  Response preview: {resp[:150]}...')

avg_adapter = sum(r['score'] for r in adapter_results) / len(adapter_results)
print(f'\n>>> ADAPTER average: {avg_adapter:.0%}')

In [ ]:
# Cell 9: Comparison summary + save results
import json

# Ollama baseline from local run (hardcoded)
ollama_results = [
    {'id': 'sqli_detect', 'score': 0.5, 'hits': ['SQL injection', 'CWE-89'], 'misses': ['parameterized', 'prepared statement']},
    {'id': 'xss_exploit', 'score': 0.0, 'hits': [], 'misses': ['<script>', 'alert', 'document.cookie', 'XSS', 'CWE-79']},
    {'id': 'path_traversal', 'score': 0.2, 'hits': ['sanitize'], 'misses': ['path traversal', 'CWE-22', 'directory traversal', 'os.path.basename']},
    {'id': 'hardening', 'score': 0.5, 'hits': ['parameterized', '?', 'execute('], 'misses': ['%s', 'bcrypt', 'hash']},
    {'id': 'taint_analysis', 'score': 0.17, 'hits': ['cookie'], 'misses': ['SSTI', 'template injection', 'render_template_string', 'CWE-94', 'Jinja']},
]
avg_ollama = sum(r['score'] for r in ollama_results) / len(ollama_results)

print('=' * 70)
print('  ATTESTOR 4.3 — MODEL BENCHMARK RESULTS')
print('=' * 70)

header = f'  {"Model":<40s} {"Avg Score":>10s}'
sep = f'  {"-"*40:<40s} {"-"*10:>10s}'
print(header)
print(sep)
print(f'  {"Ollama qwen2.5-coder:3b (local)":<40s} {avg_ollama:>9.0%}')
print(f'  {"Qwythos-9B base (no adapter)":<40s} {avg_base:>9.0%}')
print(f'  {"Qwythos-9B + trained LoRA adapter":<40s} {avg_adapter:>9.0%}')

print(f'\n  Adapter vs base delta:   {avg_adapter - avg_base:>+.0%}')
print(f'  Adapter vs Ollama delta: {avg_adapter - avg_ollama:>+.0%}')

print('\nPer-task breakdown:')
print(f'  {"Task":<20s} {"Ollama 3b":>10s} {"Qwythos base":>13s} {"Qwythos+LoRA":>13s}')
print(f'  {"-"*20:<20s} {"-"*10:>10s} {"-"*13:>13s} {"-"*13:>13s}')
for i, p in enumerate(BENCHMARK_PROMPTS):
    o = ollama_results[i]['score']
    b = base_results[i]['score']
    a = adapter_results[i]['score']
    delta = '+' if a > b else (' ' if a == b else '-')
    print(f'  {p["id"]:<20s} {o:>9.0%} {b:>12.0%} {a:>12.0%} {delta}')

# Save
all_results = {
    'ollama_qwen25_3b': ollama_results,
    'qwythos_base': base_results,
    'qwythos_adapter': adapter_results,
    'summary': {
        'ollama_avg': avg_ollama,
        'base_avg': avg_base,
        'adapter_avg': avg_adapter,
        'adapter_vs_base_delta': avg_adapter - avg_base,
        'adapter_vs_ollama_delta': avg_adapter - avg_ollama,
    }
}
with open('/kaggle/working/benchmark_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nFull results saved to /kaggle/working/benchmark_results.json')